### Fig. 4 GIMMS evaluation

Same bar and distribution figures as `evaluate_gimms.ipynb` (CDD / CDDP / SIAM / SIAMP).


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MultipleLocator
from scipy import stats

HERE = Path.cwd()
if HERE.name != "main":
    cand = Path("/Users/xingyihuang/Jupyter Code/Phenology/New Code/figures/main")
    if cand.is_dir():
        os.chdir(cand)
        HERE = cand

ROOT = HERE.parents[1]
CACHE_DIR = ROOT / "results/model/evaluation/gimms/cache"
OUT_DIR = ROOT / "results/figure4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = CACHE_DIR / "eval_results_boundexp_siam_kge08_cv5.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"Missing original GIMMS eval CSV: {csv_path}")

res = pd.read_csv(csv_path)
MODEL_ORDER = ["CDD", "CDDP", "SIAM", "SIAMP"]
res = res[res["model"].isin(MODEL_ORDER)].copy()
print("Loaded", csv_path, f"| pixels={res['pixel'].nunique()} | models={sorted(res['model'].unique())}")

model_colors = {
    "CDD": "#a3d4e1",
    "CDDP": "#5cadd8",
    "SIAM": "#ea9e87",
    "SIAMP": "#cf847e",
}
order = [m for m in MODEL_ORDER if m in set(res["model"])]
PAIRS = [("CDD", "CDDP"), ("SIAM", "SIAMP")]


def paired_ttest_p(base, pmod, metric):
    a = res.loc[res["model"] == base].set_index("pixel")[metric]
    b = res.loc[res["model"] == pmod].set_index("pixel")[metric]
    common = a.index.intersection(b.index)
    if len(common) < 3:
        return np.nan
    av = a.loc[common].to_numpy(dtype=float)
    bv = b.loc[common].to_numpy(dtype=float)
    ok = np.isfinite(av) & np.isfinite(bv)
    if ok.sum() < 3 or np.nanstd(av[ok] - bv[ok]) == 0:
        return np.nan
    _, pval = stats.ttest_rel(av[ok], bv[ok])
    return float(pval)


def stars(p):
    if not np.isfinite(p):
        return "ns"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def plot_metric_bars(col, ylabel, ylim, y_tick):
    mean_vals, err_vals = [], []
    for m in order:
        vals = res.loc[res["model"] == m, col].dropna()
        mean_vals.append(float(vals.mean()) if not vals.empty else np.nan)
        n = int(len(vals))
        std_val = float(vals.std(ddof=1)) if n > 1 else 0.0
        se = std_val / np.sqrt(n) if n > 1 and np.isfinite(std_val) else 0.0
        err_vals.append(2.0 * se)

    y_lo0, y_hi0 = ylim
    y_span0 = y_hi0 - y_lo0
    fig, ax = plt.subplots(figsize=(6, 5))
    bars = ax.bar(
        order, mean_vals,
        color=[model_colors.get(m, "#cccccc") for m in order],
    )
    xpos = {m: float(i) for i, m in enumerate(order)}
    label_y = {}
    offset = 0.02 * y_span0

    for bar, m, mean_val, err in zip(bars, order, mean_vals, err_vals):
        x = xpos[m]
        if not np.isfinite(mean_val):
            continue
        y_top = mean_val
        if err > 0:
            half_cap = bar.get_width() * 0.12
            ax.plot([x, x], [mean_val, mean_val + err], color="#333333", lw=0.9, solid_capstyle="butt", zorder=3)
            ax.plot(
                [x - half_cap, x + half_cap], [mean_val + err, mean_val + err],
                color="#333333", lw=0.9, solid_capstyle="butt", zorder=3,
            )
            y_top = mean_val + err
        ypos = y_top + offset
        label_val = np.floor(mean_val * 100.0) / 100.0
        ax.text(x, ypos, f"{label_val:.2f}", ha="center", va="bottom", fontsize=16, fontweight="bold")
        label_y[m] = ypos

    step = 0.07 * y_span0
    tick = 0.015 * y_span0
    gap = 0.16 * y_span0
    y_max = y_hi0
    for base, pmod in PAIRS:
        if base not in xpos or pmod not in xpos:
            continue
        pval = paired_ttest_p(base, pmod, col)
        s = stars(pval)
        x1, x2 = xpos[base], xpos[pmod]
        hi = max(label_y.get(base, mean_vals[order.index(base)]), label_y.get(pmod, mean_vals[order.index(pmod)]))
        y = hi + gap
        ax.plot([x1, x1, x2, x2], [y - tick, y, y, y - tick], color="#333333", lw=0.7, clip_on=False)
        ax.text(0.5 * (x1 + x2), y + tick * 0.25, s, ha="center", va="bottom", fontsize=20, color="#222222")
        print(f"{col} {base} vs {pmod}: p={pval:.2e} → {s}")
        y_max = max(y_max, y + 1.2 * step)

    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=45, ha="right", fontsize=16)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.set_ylim(y_lo0, y_max)
    ax.yaxis.set_major_locator(MultipleLocator(y_tick))
    ax.grid(False)
    ax.tick_params(axis="both", labelsize=16)
    fig.subplots_adjust(bottom=0.22, top=0.90)
    return fig


def plot_metric_distrib(col, ylabel):
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(6, 5))
    data = [res.loc[res["model"] == m, col].dropna().to_numpy() for m in order]
    bp = ax.boxplot(
        data, positions=np.arange(len(order)), widths=0.55, patch_artist=True,
        showfliers=False, medianprops=dict(color="k", lw=1.5),
    )
    for patch, m in zip(bp["boxes"], order):
        patch.set_facecolor(model_colors.get(m, "#cccccc"))
        patch.set_alpha(0.85)
    for i, (m, vals) in enumerate(zip(order, data)):
        if vals.size == 0:
            continue
        x = i + rng.uniform(-0.12, 0.12, size=vals.size)
        ax.scatter(x, vals, s=18, c="k", alpha=0.45, zorder=3, linewidths=0)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=45, ha="right", fontsize=16)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.yaxis.set_major_locator(MultipleLocator(1 if col == "rmse" else 0.1))
    ax.tick_params(axis="both", labelsize=16)
    ax.grid(False)
    fig.subplots_adjust(bottom=0.22, top=0.90)
    return fig


### Bars (RMSE, r, KGE)


In [ ]:
fig_rmse = plot_metric_bars("rmse", "RMSE (days)", ylim=(8.0, 11.0), y_tick=1)
fig_r = plot_metric_bars("r", "r", ylim=(0.15, 0.35), y_tick=0.1)
fig_kge = plot_metric_bars("kge", "KGE", ylim=(0.05, 0.25), y_tick=0.1)

stem = OUT_DIR / "eval_model_compare_gimms"
fig_rmse.savefig(str(stem) + "_rmse.png", dpi=500, bbox_inches="tight")
fig_r.savefig(str(stem) + "_r.png", dpi=500, bbox_inches="tight")
fig_kge.savefig(str(stem) + "_kge.png", dpi=500, bbox_inches="tight")
plt.show()
print("Saved", str(stem) + "_rmse.png")
print("Saved", str(stem) + "_r.png")
print("Saved", str(stem) + "_kge.png")
res.groupby("model")[["rmse", "r", "kge"]].mean().reindex(order).round(3)
